In [1]:
# inspect_linears.py
import torch
import torch.nn as nn
from pathlib import Path
from funasr import AutoModel
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
# print(torch.cuda.device(0))
print(torch.cuda.get_device_name(0))
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('NOT connected to a GPU!. Will use CPU!')
else:
  print('Connected to a GPU!')
# load/create your model exactly as in training
# e.g. from mymodel import Paraformer
# model = Paraformer(**model_args)
# ckpt = torch.load("exp/epoch-best.pt", map_location="cpu")
# model.load_state_dict(ckpt["model_state_dict"], strict=False)

# model = AutoModel(model="stream_bangla")

True
1
0
NVIDIA A100-SXM4-80GB
Connected to a GPU!


In [12]:
import os
print(os.getcwd())


/home/ubuntu/FunASR


In [13]:
import zipfile
with zipfile.ZipFile("train_cv.zip", 'r') as zip_ref:
    zip_ref.extractall('./')

with zipfile.ZipFile("stream_bangla.zip", 'r') as zip_ref:
    zip_ref.extractall('./')

In [2]:
# {'attention_heads': 4
# 'linear_units': 2048
# 'num_blocks': 16
# 'dropout_rate': 0.1
# 'positional_dropout_rate': 0.1
# 'self_attention_dropout_rate': 0.1
# 'src_attention_dropout_rate': 0.1
# 'att_layer_num': 16
# 'kernel_size': 11
# 'sanm_shfit': 5
# 'vocab_size': 500}

In [2]:
model.model

ParaformerStreaming(
  (specaug): SpecAugLFR(
    (freq_mask): MaskAlongAxisLFR(mask_width_range=[0, 30], num_mask=1, axis=freq)
    (time_mask): MaskAlongAxisLFR(mask_width_range=[0, 12], num_mask=1, axis=time)
  )
  (encoder): SANMEncoderChunkOpt(
    (embed): StreamSinusoidalPositionEncoder()
    (encoders0): MultiSequential(
      (0): EncoderLayerSANM(
        (self_attn): MultiHeadedAttentionSANM(
          (linear_out): Linear(in_features=512, out_features=512, bias=True)
          (linear_q_k_v): Linear(in_features=560, out_features=1536, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fsmn_block): Conv1d(512, 512, kernel_size=(11,), stride=(1,), groups=512, bias=False)
          (pad_fn): ConstantPad1d(padding=(5, 5), value=0.0)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropo

In [11]:
type(model.model.decoder.embed)

torch.nn.modules.container.Sequential

In [3]:
model.model.decoder.embed = nn.Sequential(nn.Embedding(118, 512))

In [4]:
model.model.decoder.output_layer = nn.Linear(512, 118).to('cuda:0')

In [5]:
model.model

ParaformerStreaming(
  (specaug): SpecAugLFR(
    (freq_mask): MaskAlongAxisLFR(mask_width_range=[0, 30], num_mask=1, axis=freq)
    (time_mask): MaskAlongAxisLFR(mask_width_range=[0, 12], num_mask=1, axis=time)
  )
  (encoder): SANMEncoderChunkOpt(
    (embed): StreamSinusoidalPositionEncoder()
    (encoders0): MultiSequential(
      (0): EncoderLayerSANM(
        (self_attn): MultiHeadedAttentionSANM(
          (linear_out): Linear(in_features=512, out_features=512, bias=True)
          (linear_q_k_v): Linear(in_features=560, out_features=1536, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fsmn_block): Conv1d(512, 512, kernel_size=(11,), stride=(1,), groups=512, bias=False)
          (pad_fn): ConstantPad1d(padding=(5, 5), value=0.0)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropo

118 tokens bangla and english

In [6]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 211591383


In [7]:
model.generate("imtc_2_ken_9621_Q18.mp3", batch_size_s=300)

rtf_avg: 2.173: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.53it/s]


[{'key': 'imtc_2_ken_9621_Q18', 'text': "8স ৪য় -িFC ' ' ' ' e3৭ঋ"}]

In [27]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 211675433


In [15]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 215884085


In [61]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 220084533


In [2]:
torchrun \
--nnodes 1 \
--nproc_per_node 0 \
funasr/bin/train.py \
--config-path "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR" \
--config-name bangla_streamer \
++train_data_set_list="audio_train_datasets" \
++valid_data_set_list="audio_val_datasets" \
++tokenizer_conf.token_list="C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\stream_bangla\tokens.json" \
++frontend_conf.cmvn_file="am.mvn" \
++dataset_conf.batch_size=3 \
++dataset_conf.batch_type="example" \
++dataset_conf.num_workers=4 \
++train_conf.max_epoch=150 \
++optim_conf.lr=0.0002 \
++init_param="bangla_asr_paraformer_streaming_model.pt" \
++output_dir="./"

SyntaxError: invalid syntax (2522301627.py, line 2)

In [ ]:
import torch
import os

# --- SETTINGS ---
# CHANGE THIS to match your actual checkpoint name
input_file = "outputs/model.pt.best" 
output_file = "outputs/bangla_streaming_final.pt"

print(f"📦 Loading heavy checkpoint: {input_file}...")

try:
    # 1. Load the massive 2.4GB file (CPU to save VRAM)
    checkpoint = torch.load(input_file, map_location="cpu")
    
    # 2. Inspect and Strip
    # FunASR checkpoints usually have the keys: 'state_dict', 'optimizer', 'scheduler', etc.
    print(f"   Keys found inside: {list(checkpoint.keys())}")
    
    if 'optimizer' in checkpoint:
        print("   ✂️  Deleting 'optimizer' (approx 1.6GB)...")
        del checkpoint['optimizer']
        
    if 'scheduler' in checkpoint:
        print("   ✂️  Deleting 'scheduler'...")
        del checkpoint['scheduler']

    if 'criterion' in checkpoint:
        del checkpoint['criterion']
        
    # 3. Save the clean file
    print(f"💾 Saving slim model to: {output_file}...")
    torch.save(checkpoint, output_file)
    
    # 4. Compare Sizes
    orig_mb = os.path.getsize(input_file) / 1024 / 1024
    new_mb = os.path.getsize(output_file) / 1024 / 1024
    
    print("\n✅ DONE!")
    print(f"   Original Size: {orig_mb:.2f} MB")
    print(f"   Final Size:    {new_mb:.2f} MB")
    print(f"   You can now download '{output_file}' for your chatbot.")

except FileNotFoundError:
    print(f"❌ Error: Could not find file '{input_file}'. Check the name in your outputs folder.")
except Exception as e:
    print(f"❌ Error: {e}")

In [3]:
import shutil
import os

# 1. Define paths (Using your structure)
project_dir = r"C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR"
os.chdir(project_dir)

source_config = "FunASR/bangla_streamer.yaml"
target_config = "FunASR/stream_bangla/config.yaml"

# 2. Copy the config so FunASR recognizes the folder as a model
# This effectively "Registers" your local folder as a valid model
if os.path.exists(source_config):
    shutil.copy(source_config, target_config)
    print(f"Success! Created local model config at: {target_config}")
else:
    print(f"Error: Could not find {source_config}. Check file name.")

Success! Created local model config at: FunASR/stream_bangla/config.yaml


In [1]:
import os

if os.path.exists("outputs"):
    print("Found old 'outputs' folder. Deleting it to prevent conflicts...")
    shutil.rmtree("outputs")
    print("✅ outputs folder deleted. Starting fresh.")
else:
    print("No old outputs folder found.")

No old outputs folder found.


In [3]:
import os

# 1. Set Dir (Safe zone inside the project)
project_dir = r"C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR"
try:
    os.chdir(project_dir)
    print(f"Working Directory: {os.getcwd()}")
except FileNotFoundError:
    print("Error: Project path not found.")

# 2. Set Env Var
os.environ["USE_LIBUV"] = "0"
print("Starting")
# 3. RUN COMMAND (Using RELATIVE paths for datasets)
# I changed the long C:\... paths to just "FunASR/..."
!python -m funasr.bin.train ++model="damo/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-online" ++config-path="FunASR" ++config-name=bangla_streamer ++train_data_set_list="FunASR/audio_train_datasets.jsonl" ++valid_data_set_list="FunASR/audio_val_datasets.jsonl" ++tokenizer_conf.token_list="FunASR/stream_bangla/tokens.json" ++frontend_conf.cmvn_file="FunASR/stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true
# !python -m funasr.bin.train ++model="FunASR/stream_bangla" ++init_param="FunASR/stream_bangla/model.pt" ++config-path="FunASR" ++config-name=bangla_streamer ++train_data_set_list="FunASR/audio_train_datasets.jsonl" ++valid_data_set_list="FunASR/audio_val_datasets.jsonl" ++tokenizer_conf.token_list="FunASR/stream_bangla/tokens.json" ++frontend_conf.cmvn_file="FunASR/stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true

Working Directory locked to: C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR
Starting
[2025-12-11 22:36:29,486][root][INFO] - download models from model hub: ms
[2025-12-11 22:36:29,495][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                                                                                     |
| AudioDataset             | AudioDataset             | C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\llm_datasets\datasets.py:302         |

Error executing job with overrides: ['++model=stream_bangla', '++config-path=stream_bangla', '++config-name=config.yaml', '++train_data_set_list=audio_train_datasets.jsonl', '++valid_data_set_list=audio_val_datasets.jsonl', '++tokenizer_conf.token_list=stream_bangla/tokens.json', '++frontend_conf.cmvn_file=stream_bangla/am.mvn', '++dataset_conf.batch_size=3', '++dataset_conf.num_workers=0', '++output_dir=outputs', '++ignore_init_mismatch=true']

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\bin\train.py", line 265, in <module>
    main_hydra()
  File "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\ASR\Lib\site-packages\hydra\main.py", line 94, in decorated_main
    _run_hydra(
  File "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\ASR\Lib\site-packages\hydra\_internal\utils.py", line 394, in _run_hydra

In [4]:
import torch
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Capability: {torch.cuda.get_device_capability(0)}")

# Try a small tensor operation on GPU to prove it works
try:
    x = torch.tensor([1.0]).cuda()
    print("✅ Success! GPU is working.")
except Exception as e:
    print(f"❌ Error: {e}")

Torch: 2.9.1+cu126
CUDA: 12.6
GPU: NVIDIA GeForce RTX 5080
Capability: (12, 0)
✅ Success! GPU is working.


C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5080 which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (5.0) - (9.0)
    
  warnings.warn(
C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 13.0 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:326: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeFor

In [3]:
!pip uninstall -y flash-attn


In [2]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version used by Torch: {torch.version.cuda}")


PyTorch Version: 2.9.1+cu126
CUDA Version used by Torch: 12.6


In [1]:
import os
import shutil

# 1. Env Variables to fix "No Kernel Image"
os.environ["USE_LIBUV"] = "0"
os.environ["TORCH_CUDNN_V8_API_ENABLED"] = "1" # Forces newer cuDNN backend
# Disable specific optimizations that might crash on new hardware
os.environ["WET_DISABLE_FLASH_ATTN"] = "1" 

# 2. Setup
project_dir = r"C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR"
try:
    os.chdir(project_dir)
except:
    pass

# Cleanup
if os.path.exists("outputs"):
    shutil.rmtree("outputs")

print("🚀 Starting Training (Safe Mode)...")

# 3. Run Command
!python -m funasr.bin.train ++model="stream_bangla" ++config-path="stream_bangla" ++config-name='config.yaml' ++train_data_set_list="audio_train_datasets.jsonl" ++valid_data_set_list="audio_val_datasets.jsonl" ++tokenizer_conf.token_list="stream_bangla/tokens.json" ++frontend_conf.cmvn_file="stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true

🚀 Starting Training (Safe Mode)...
[2025-12-12 05:48:11,693][root][INFO] - download models from model hub: ms
[2025-12-12 05:48:11,703][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                                                                                 |
| AudioDataset             | AudioDataset             | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\llm_datasets\datasets.py:302         |
| AudioLLMDataset          | AudioLLMDataset          | C:\MenoChat\code\code\Me

Error executing job with overrides: ['++model=stream_bangla', '++config-path=stream_bangla', "++config-name='config.yaml'", '++train_data_set_list=audio_train_datasets.jsonl', '++valid_data_set_list=audio_val_datasets.jsonl', '++tokenizer_conf.token_list=stream_bangla/tokens.json', '++frontend_conf.cmvn_file=stream_bangla/am.mvn', '++dataset_conf.batch_size=3', '++dataset_conf.num_workers=0', '++output_dir=outputs', '++ignore_init_mismatch=true']

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\bin\train.py", line 265, in <module>
    main_hydra()
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\main.py", line 94, in decorated_main
    _run_hydra(
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\_internal\utils.py", line 394, in _run_hydra
    _run_

In [ ]:
Working

In [4]:
!cp outputs/model.pt.best stream_bangla_A100

In [1]:
!rm -rf funasr/utils/__pycache__/

In [5]:
from funasr import AutoModel
import os

print("Done")
model = AutoModel(
    model="stream_bangla_A100", 
    disable_update=True,

)

# Pick a test file
test_wav = "/test_cv/common_voice_bn_31645863.mp3" # <--- REPLACE THIS

res = model.generate(input=test_wav)
print(f"\n🤖 Model Says: {res}")

Done
funasr version: 1.2.9.


  0%|          | 0/1 [00:00<?, ?it/s]

TackTack
TackTack
TackTack
TackTack
TackTack
TackTack
TackTack
TackTack
TackTack
TackTack
TackTack
TackTack


rtf_avg: 2.176: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]                                                                                          

TackTack

🤖 Model Says: [{'key': 'common_voice_bn_31645863', 'text': 'সোমানের মাংস চ ও ওাম নার জন্য এই   প্র প্র প্র চিতদ্ধ ।'}]


In [6]:
!git --version

git version 2.34.1


In [ ]:
!git config --global user.name "King-Rafat"
!git config --global user.email "kingrafat82@gmail.com"

In [5]:
import sentencepiece as spm

bpe_model = "stream_bangla_A100/bpe.model" 
tokenizer = spm.SentencePieceProcessor(model_file=bpe_model)


In [ ]:
[{'key': 'common_voice_bn_31628801', 'text': 'জটি কেরলভূভূ ছাড়াায়াঅন্যান্য পশ্চিমপশ্চিম পাড়িকেএকটিত্রীুবন্দ দপ্রস্তাববস্বাগত্ব জানত।'}]


In [7]:
ids = model.generate(input=test_wav, output_token_ids=True)
print("Done")
print(ids)
print([tokenizer.decode[i] for i in ids])

  0%|          | 0/1 [00:00<?, ?it/s]

জটি কেরল
ভূভূ ছাড়াায়া
অন্যান্য পশ্চিম
পশ্চিম পাড়িকে
একটিত্রীুবন্দ দ


rtf_avg: 1.994: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]                                                                                          

প্রস্তাবব
স্বাগত্ব জানত
।
This used!
Done
[{'key': 'common_voice_bn_31628801', 'text': 'জটি কেরলভূভূ ছাড়াায়াঅন্যান্য পশ্চিমপশ্চিম পাড়িকেএকটিত্রীুবন্দ দপ্রস্তাববস্বাগত্ব জানত।'}]


TypeError: 'method' object is not subscriptable

In [3]:
token_ids = model.predict(test_wav)
tokens = tokenizer.ids2tokens(token_ids)
print(tokens)

AttributeError: 'AutoModel' object has no attribute 'predict'

In [7]:
!grep -R "ids2tokens" funasr/models/paraformer* -n

grep: funasr/models/paraformer/__pycache__/model.cpython-311.pyc: binary file matches
funasr/models/paraformer/model.py:569:                    token = tokenizer.ids2tokens(token_int)
grep: funasr/models/paraformer_streaming/__pycache__/model.cpython-311.pyc: binary file matches
funasr/models/paraformer_streaming/model.py:547:                token = tokenizer.ids2tokens(token_int)


In [21]:
raw_output = res[0]['text']
clean_output = raw_output.replace(" ", "").replace("▁", " ").replace(" ", " ").strip()

print(f"\n🤖 Raw: {raw_output}")
print(f"✅ Clean: {clean_output}")


🤖 Raw: জ ট িত িক  আ ল োঅ ন ্য ভ ূ ছ াড ়াায ়অ ন ্য ান ্য প শ ্চ িম ং প াগ েট ্র ত ্র ীব িব িয ়ে দ প ্র  প ্র স ্ত াব ক েস ্ব াগ ত  ঝ  জ ান ান ।
✅ Clean: জটিতিকআলোঅন্যভূছাড়াায়অন্যান্যপশ্চিমংপাগেট্রত্রীবিবিয়েদপ্রপ্রস্তাবকেস্বাগতঝজানান।


In [1]:
from funasr import AutoModel
import sentencepiece as spm
import os

# 1. SETUP
# Ensure these point to your actual files
model_dir = "outputs"
bpe_model = "stream_bangla_A100/bpe.model" 

# 2. LOAD BPE PROCESSOR
# We load this to do the decoding ourselves
sp = spm.SentencePieceProcessor(model_file=bpe_model)

# 3. LOAD MODEL
model = AutoModel(model=model_dir, disable_update=True)

# 4. INFERENCE
test_wav = "/test_cv/common_voice_bn_31628801.mp3" # <--- REPLACE THIS
res = model.generate(input=test_wav)

# 5. THE RAW OUTPUT
raw_text = res[0]['text']
print(f"RAW TOKENS: {raw_text}")

# 6. THE CORRECT DECODE
# FunASR adds a space between every token. We split by space to get the list back.
# Note: If your model actually output a "Space" token (not underscore), split() removes it.
# But for standard BPE (which uses ▁), this works perfectly.
token_list = raw_text.split()

# Ask SentencePiece to glue it back together
clean_text = sp.decode(token_list)

print(f"FINAL TEXT: {clean_text}")

funasr version: 1.2.9.


rtf_avg: 5.573: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]                                                                                          

RAW TOKENS: জ ট িত িক  আ ল োঅ ন ্য ভ ূ ছ াড ়াায ়অ ন ্য ান ্য প শ ্চ িম ং প াগ েট ্র ত ্র ীব িব িয ়ে দ প ্র  প ্র স ্ত াব ক েস ্ব াগ ত  ঝ  জ ান ান ।
FINAL TEXT: জটিতিকআলোঅন্যভূছাড়াায়অন্যান্যপশ্চিমংপাগেট্রত্রীবিবিয়েদপ্রপ্রস্তাবকেস্বাগতঝজানান।


In [7]:
!grep -R "sentence_postprocess_sentencepiece" -n funasr/

funasr/utils/.ipynb_checkpoints/postprocess_utils-checkpoint.py:254:def sentence_postprocess_sentencepiece(words):
grep: funasr/utils/__pycache__/postprocess_utils.cpython-311.pyc: binary file matches
funasr/utils/postprocess_utils.py:254:def sentence_postprocess_sentencepiece(words):


In [11]:
def postprocess_bn_asr(text):
    text = text.strip()
    text = text.replace(" ", "")
    text = unicodedata.normalize("NFC", text)
    return text
postprocess_bn_asr(res[0]['text'])

'জটিকেরলভূভূছাড়াায়াঅন্যান্যপশ্চিমপশ্চিমপাড়িকেএকটিবিুবন্দদপ্রস্তাববস্বাগত্বজানত।'

In [10]:
import unicodedata

def normalize_bn(text):
    return unicodedata.normalize("NFC", text)

raw = "জ ট ি ক ের ল ভ ূভ ূ ছ াড ়াায ়াঅ ন ্য ান ্য"
print(normalize_bn(raw))


জ ট ি ক ের ল ভ ূভ ূ ছ াড ়াায ়াঅ ন ্য ান ্য


In [2]:
res[0]['text']

'জ ট িত িক  আ ল োঅ ন ্য ভ ূ ছ াড ়াায ়অ ন ্য ান ্য প শ ্চ িম ং প াগ েট ্র ত ্র ীব িব িয ়ে দ প ্র  প ্র স ্ত াব ক েস ্ব াগ ত  ঝ  জ ান ান ।'

In [3]:
def fix_bpe_output(text):
    # 1. Remove the spaces FunASR added
    # 2. Replace the special BPE underscore ' ' (U+2581) with a real space
    # 3. Strip extra whitespace
    return text.replace(" ", "").replace(" ", " ").strip()

print(f"FIXED: {fix_bpe_output(res[0]['text'])}")

FIXED: জটিতিকআলোঅন্যভূছাড়াায়অন্যান্যপশ্চিমংপাগেট্রত্রীবিবিয়েদপ্রপ্রস্তাবকেস্বাগতঝজানান।


In [6]:
def fix_bangla_output(text):
    # Step 1: Remove the artificial spaces FunASR added between every character
    clean = text.replace(" ", "")
    
    # Step 2: If your model predicted a special token for space (e.g. <s> or |), replace it here.
    # If your vocab DOES NOT have a space token, your model output will be one long string.
    # (e.g. 'তিনিএকটি...')
    return clean

print(f"FIXED: {fix_bangla_output(res[0]['text'])}")

FIXED: তিনিএকটিসোযেরজন্যবক্স্ানহহযেভররতেআসেন


In [2]:
from funasr import AutoModel
import os

print("🕵️ INSPECTING LOADED TOKENIZER...")

# 1. Load the model
# model_path = "outputs" 
model = AutoModel(model="stream_bangla", disable_update=True)

# 2. Get the actual Tokenizer object from memory
# (Note: In AutoModel, it's usually stored in model.kwargs['tokenizer'] or internal attributes)
# We will hunt for it.
tokenizer = None

# Try standard location
if hasattr(model, 'tokenizer'):
    tokenizer = model.tokenizer
# Try digging into the kwargs if the above is None/Wrapper
elif hasattr(model, 'kwargs') and 'tokenizer' in model.kwargs:
    tokenizer = model.kwargs['tokenizer']

if tokenizer is None:
    print("❌ Could not find tokenizer object directly. Dumping model attributes:")
    print(dir(model))
else:
    print(f"\n✅ Tokenizer Class: {type(tokenizer)}")
    print(f"   (Should be <class 'funasr.tokenizer.char_tokenizer.CharTokenizer'>)")

    # 3. Check the "Killer" Attributes
    print("\n🔍 CRITICAL CONFIG CHECK:")
    
    # Check seg_dict (The Chinese Cleaner)
    if hasattr(tokenizer, 'seg_dict'):
        print(f"   seg_dict: {tokenizer.seg_dict}")
        if tokenizer.seg_dict is None:
            print("   ✅ seg_dict is NONE (Safe).")
        else:
            print("   ❌ seg_dict is ACTIVE! (This explains 99% accuracy issues if present)")
    
    # Check split_with_space
    if hasattr(tokenizer, 'split_with_space'):
        print(f"   split_with_space: {tokenizer.split_with_space}")
        
    # Check the actual vocab size loaded in memory
    if hasattr(tokenizer, 'token_list'):
        print(f"   Vocab Size: {len(tokenizer.token_list)}")
        print(f"   First 10 tokens: {tokenizer.token_list[:10]}")
        
        # Check for SPACE
        has_space = " " in tokenizer.token_list or "<space>" in tokenizer.token_list
        print(f"   Can it predict Space? {has_space}")

🕵️ INSPECTING LOADED TOKENIZER...
funasr version: 1.2.9.


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [2]:
!curdir

/bin/bash: line 1: curdir: command not found


In [3]:
!rm -r outputs

In [1]:
import torch
import os # <--- 1. Import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

# --- CHANGE 1: CRITICAL FOR BANGLA ON WINDOWS ---
# Without this, 'tokens.json' is read as CP1252 (Western) instead of UTF-8.
# The model will see every Bangla character as <unk> and learn nothing.
os.environ["PYTHONUTF8"] = "1" 

# --- CHANGE 2: NETWORK SAFETY ---
# Prevents random crashes with torch distributed on Windows
os.environ["USE_LIBUV"] = "0"

# --- YOUR COMMAND (Unchanged) ---
!python -m funasr.bin.train ++model="stream_bangla_A100" ++config-path="stream_bangla_A100" ++config-name='config.yaml' ++train_data_set_list="audio_train_datasets.jsonl" ++valid_data_set_list="audio_test_datasets.jsonl" ++frontend_conf.cmvn_file="stream_bangla_A100/am.mvn" ++dataset_conf.batch_size=24 ++dataset_conf.num_workers=8 ++output_dir="outputs" ++ignore_init_mismatch=true

PyTorch Version: 2.4.1+cu124
CUDA Version: 12.4
Device Name: NVIDIA A100-SXM4-80GB
[2026-01-02 12:17:52,670][root][INFO] - download models from model hub: ms
[2026-01-02 12:17:52,737][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                       |
| AudioDataset             | AudioDataset             | funasr/datasets/audio_datasets/datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | funasr/datasets/audio_datasets/datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | funasr/datasets/llm_datasets/datasets.py:302         |
| AudioLLMDataset          | AudioLLMDataset          | funasr/datasets/llm_datasets/datasets.py:167         |
| AudioLLMNARDataset       | AudioLLMNARDataset       | funasr/datasets/llm_datasets/datasets.py:8           |
| AudioLLMQwenAudioDataset | AudioLLMQwenA

In [2]:
import inspect
from funasr.tokenizer.sentencepiece_tokenizer import SentencepiecesTokenizer

# Get the list of arguments the class expects
sig = inspect.signature(SentencepiecesTokenizer.__init__)
print("Arguments required:", sig.parameters.keys())

Arguments required: odict_keys(['self', 'bpemodel', 'kwargs'])


In [3]:
import os
import json
import yaml
from funasr.tokenizer.char_tokenizer import CharTokenizer

# 1. Setup Paths
model_dir = "stream_bangla"
token_path = os.path.join(model_dir, "tokens.json")
config_path = os.path.join(model_dir, "config.yaml")

print(f"🔧 Debugging Tokenizer...")

# 2. Manually Load the Vocabulary (The Manual Fix)
if not os.path.exists(token_path):
    print("❌ Error: tokens.json not found!")
    exit()

with open(token_path, 'r', encoding='utf-8') as f:
    vocab_list = json.load(f)

print(f"✅ Loaded {len(vocab_list)} tokens from JSON.")

# 3. Create the Mapping Manually
# FunASR failed to do this, so we do it ourselves to test
token2id = {token: i for i, token in enumerate(vocab_list)}

# 4. Initialize Tokenizer & Inject Data
tokenizer = CharTokenizer(symbol_table=token_path)
tokenizer.token2id = token2id       # <--- INJECTING THE MISSING ATTRIBUTE
tokenizer.token_list = vocab_list   # <--- INJECTING THE MISSING LIST
tokenizer.unk_id = token2id.get('<unk>', 1)

print("✅ Tokenizer patched manually.")

# 5. TEST THE TEXT CLEANING LOGIC
# Now that the map works, we can finally see if the Config is deleting Bangla
text = "আমি বাংলা"
print(f"\n📝 Input: '{text}'")

try:
    ids = tokenizer.encode(text)
    print(f"🔢 IDs:   {ids}")
    
    # Check results
    if len(ids) == 0:
        print("\n❌ RESULT: EMPTY LIST.")
        print("   This means the 'Chinese Cleaner' in config.yaml is deleting your text.")
    elif all(x == tokenizer.unk_id for x in ids):
        print(f"\n❌ RESULT: ALL <unk> (ID: {tokenizer.unk_id})")
        print("   The text survived, but does not match the vocab list.")
    else:
        decoded = [vocab_list[i] for i in ids]
        print(f"🔤 Decoded: {decoded}")
        print("\n✅ SUCCESS: The tokenizer works!")
        
except Exception as e:
    print(f"❌ Still crashing: {e}")

# --- 6. CHECK CONFIG FOR THE ROOT CAUSE ---
# If the above printed "EMPTY LIST", we need to check the config file
print("\n🔍 Checking config.yaml for the 'Killer' setting...")
with open(config_path, 'r', encoding='utf-8') as f:
    conf = yaml.safe_load(f)
    t_conf = conf.get('tokenizer_conf', {})
    
    seg_dict = t_conf.get('seg_dict', None)
    
    if seg_dict is not None:
        print(f"❌ FOUND THE CULPRIT: seg_dict = {seg_dict}")
        print("   You MUST delete this line from config.yaml!")
    else:
        print("✅ seg_dict is None (Good).")

🔧 Debugging Tokenizer...
✅ Loaded 118 tokens from JSON.
✅ Tokenizer patched manually.

📝 Input: 'আমি বাংলা'
🔢 IDs:   [5, 39, 56, 37, 55, 51, 42, 55]
🔤 Decoded: ['আ', 'ম', 'ি', 'ব', 'া', 'ং', 'ল', 'া']

✅ SUCCESS: The tokenizer works!

🔍 Checking config.yaml for the 'Killer' setting...
✅ seg_dict is None (Good).


In [1]:
# import torch

# print(f"PyTorch Version: {torch.__version__}")
# # You want to see something like "2.6.0.dev2025..." or higher

# print(f"CUDA Version: {torch.version.cuda}")
# # You want to see "12.6" or "12.8"

# print(f"Device Name: {torch.cuda.get_device_name(0)}")

# !python -m funasr.bin.train ++model="stream_bangla" ++config-path="stream_bangla" ++config-name='config.yaml' ++train_data_set_list="audio_train_datasets.jsonl" ++valid_data_set_list="audio_test_datasets.jsonl" ++tokenizer_conf.token_list="stream_bangla/tokens.json" ++frontend_conf.cmvn_file="stream_bangla/am.mvn" ++dataset_conf.batch_size=16 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true

PyTorch Version: 2.5.1+cu121
CUDA Version: 12.1
Device Name: NVIDIA A100-SXM4-40GB
[2025-12-16 17:32:50,039][root][INFO] - download models from model hub: ms
[2025-12-16 17:32:50,078][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                       |
| AudioDataset             | AudioDataset             | funasr/datasets/audio_datasets/datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | funasr/datasets/audio_datasets/datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | funasr/datasets/llm_datasets/datasets.py:302         |
| AudioLLMDataset          | AudioLLMDataset          | funasr/datasets/llm_datasets/datasets.py:167         |
| AudioLLMNARDataset       | AudioLLMNARDataset       | funasr/datasets/llm_datasets/datasets.py:8           |
| AudioLLMQwenAudioDataset | AudioLLMQwenA

In [ ]:
Working

In [1]:
import os

# 2. Env
os.environ["USE_LIBUV"] = "0"
print("starting")
# 3. RUN COMMAND
# - ++hydra.job.chdir=false : Prevents path breaking
# - ++init_param : Explicitly points to the weights
!python -m funasr.bin.train ++model="stream_bangla" ++config-path="stream_bangla" ++config-name='config.yaml' ++train_data_set_list="audio_train_datasets.jsonl" ++valid_data_set_list="audio_val_datasets.jsonl" ++tokenizer_conf.token_list="stream_bangla/tokens.json" ++frontend_conf.cmvn_file="stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true 

starting
[2025-12-12 05:18:22,138][root][INFO] - download models from model hub: ms
[2025-12-12 05:18:22,153][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                                                                                 |
| AudioDataset             | AudioDataset             | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\llm_datasets\datasets.py:302         |
| AudioLLMDataset          | AudioLLMDataset          | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR

Error executing job with overrides: ['++model=stream_bangla', '++config-path=stream_bangla', "++config-name='config.yaml'", '++train_data_set_list=audio_train_datasets.jsonl', '++valid_data_set_list=audio_val_datasets.jsonl', '++tokenizer_conf.token_list=stream_bangla/tokens.json', '++frontend_conf.cmvn_file=stream_bangla/am.mvn', '++dataset_conf.batch_size=3', '++dataset_conf.num_workers=0', '++output_dir=outputs', '++ignore_init_mismatch=true']

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\bin\train.py", line 265, in <module>
    main_hydra()
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\main.py", line 94, in decorated_main
    _run_hydra(
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\_internal\utils.py", line 394, in _run_hydra
    _run_

In [ ]:
import os
import shutil

# 1. Setup Project Directory (Fixes the parentheses path issue)
project_dir = r"C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR"
try:
    os.chdir(project_dir)
    print(f"Working Directory locked to: {os.getcwd()}")
except:
    pass

# # 2. DELETE OUTPUTS (Crucial to remove broken cache)
# if os.path.exists("outputs"):
#     shutil.rmtree("outputs")

# 3. Set Env
os.environ["USE_LIBUV"] = "0"

# 4. RUN COMMAND
# - We use RELATIVE paths (FunASR/...) to avoid the "(2)" error.
# - We add ++hydra.job.chdir=false to stop Hydra from breaking those paths.
# - We use ++init_param to load your local checkpoint.
cmd = (
    'python -m funasr.bin.train '
    '++model="stream_bangla" '
    '++model="stream_bangla" '
    '++config-path="stream_bangla/config.yaml" '
    '++config-name=config '
    '++train_data_set_list="audio_train_datasets.jsonl" '
    '++valid_data_set_list="audio_val_datasets.jsonl" '
    '++tokenizer_conf.token_list="stream_bangla/tokens.json" '
    '++frontend_conf.cmvn_file="stream_bangla/am.mvn" '
    '++dataset_conf.batch_size=3 '
    '++dataset_conf.num_workers=0 '
    '++output_dir="outputs" '
    '++ignore_init_mismatch=true '
    '++hydra.job.chdir=false'
)
print("runnin")
!{cmd}

Working Directory locked to: C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR
